# Stochastic dataset generation

End-to-end batch run of the **stochastic** `OpticalSimulator` with the SIREN TOF sampler:

1. jaxtpc front end → `(positions_mm, photons, t0_ns)` per event
2. **Voxelize at 20 mm** to keep N tractable (~20× reduction, lossless above corr ≈ 0.999)
3. Stochastic simulate → `SlicedWaveform` per event
4. Inspect total-PE distribution and a per-event waveform
5. Benchmark cell: per-event wall-clock + peak GPU memory

In [ ]:
import os, sys, time, gc
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
sys.path.insert(0, '..')
sys.path.insert(0, '../jaxtpc')

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm

import jax
import jax.numpy as jnp

from tools.geometry import generate_detector
from tools.simulation import DetectorSimulator
from tools.loader import load_event

from goop import (
    OpticalSimConfig,
    OpticalSimulator,
    Response,
    SERKernel,
    create_siren_tof_sampler,
    voxelize,
)
from goop.delays import Delays

torch.manual_seed(0)
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('torch device:', DEVICE, '| jax devices:', jax.devices())

## 1. jaxtpc front end

Build the JAX `DetectorSimulator` that turns edepsim deposits into `(positions_mm, photons, t0_ns)` per detector volume, then JIT-warm on event 0.

In [ ]:
DATA_PATH   = '/sdf/home/y/youngsam/sw/dune/sirentv/data/out.h5'
CONFIG_PATH = '../jaxtpc/config/cubic_wireplane_config.yaml'

detector_config = generate_detector(CONFIG_PATH)
jx_sim = DetectorSimulator(
    detector_config,
    total_pad=250_000,
    response_chunk_size=50_000,
    include_track_hits=False,
)
jx_cfg = jx_sim.config

# JIT warm-up so subsequent events don't pay compile time.
_warm = jx_sim.process_event_light(load_event(DATA_PATH, jx_cfg, event_idx=0))
jax.block_until_ready(_warm.volumes[0].charge)
del _warm
print(f'jaxtpc ready ({jx_cfg.n_volumes} volumes)')

## 2. Build the stochastic simulator (SIREN sampler)

Same SIREN checkpoint used elsewhere; `tick_ns=1`, no SER jitter / baseline noise so each event's PE distribution is shot-noise-only. Add `ScintillationBiexponentialDelay`, TPB re-emission, and PMT TTS via `create_default_delays()` to keep the time spectrum realistic.

In [ ]:
from goop import create_default_delays

TICK_NS = 1.0
GAIN    = -45.0

siren = create_siren_tof_sampler(device=str(DEVICE))

cfg = OpticalSimConfig(
    tof_sampler=siren,
    delays=create_default_delays(),
    kernel=Response(
        kernels=[SERKernel(duration_ns=2000.0, device=DEVICE)],
        tick_ns=TICK_NS, device=DEVICE,
    ),
    device=str(DEVICE),
    tick_ns=TICK_NS,
    gain=GAIN,
)
sim = OpticalSimulator(cfg)
print(f'stoch sim ready, n_channels = {cfg.n_channels}')

## 3. Per-event helper

Loads volume 0 of one event, voxelizes at 20 mm, runs the stochastic sim, returns the desliced dense `Waveform` plus per-event metadata (n raw segments, n voxels, total PE).

In [ ]:
VOXEL_DX = 20.0  # mm

def load_voxelized(event_idx, vol_idx=0, dx=VOXEL_DX):
    """edepsim event -> (pos_t, nph_t, tns_t) on the GPU, voxelized."""
    deposits = load_event(DATA_PATH, jx_cfg, event_idx=event_idx)
    filled   = jx_sim.process_event_light(deposits)
    jax.block_until_ready(filled.volumes[vol_idx].charge)

    vol = filled.volumes[vol_idx]
    n   = int(vol.n_actual)
    pos_np = np.asarray(vol.positions_mm[:n], dtype=np.float32)
    nph_np = np.asarray(jnp.ceil(vol.photons[:n]).astype(jnp.int32))
    tns_np = np.asarray(vol.t0_us[:n], dtype=np.float32) * 1000.0  # us -> ns

    # Voxelize on host, transfer voxelized arrays only.
    pos_v, nph_v, tns_v = voxelize(pos_np, nph_np, tns_np, dx=dx)
    pos_t = torch.from_numpy(pos_v).to(DEVICE)
    nph_t = torch.from_numpy(nph_v.astype(np.int64)).to(DEVICE)
    tns_t = torch.from_numpy(tns_v).to(DEVICE)

    del deposits, filled, vol
    jax.clear_caches(); gc.collect()
    return pos_t, nph_t, tns_t, n, len(pos_v)

def simulate_event(event_idx, vol_idx=0):
    pos_t, nph_t, tns_t, n_raw, n_vox = load_voxelized(event_idx, vol_idx)
    sw = sim.simulate(pos_t, nph_t, tns_t,
                      stitched=True, subtract_t0=True,
                      add_baseline_noise=False)
    return sw, n_raw, n_vox

# Smoke test on event 0
sw0, n_raw0, n_vox0 = simulate_event(0)
wf0 = sw0.deslice()
print(f'event 0: {n_raw0:,} -> {n_vox0:,} voxels  (reduction {n_raw0/n_vox0:.1f}x)')
print(f'  waveform shape: {tuple(wf0.adc.shape)}, t0={wf0.t0:.1f} ns, peak |adc|={wf0.adc.abs().max().item():.1f}')

## 4. Batch run

Loop over a handful of events. We keep the desliced `Waveform` for inspection of one and just record per-PMT total |ADC| (≈ PE×gain) for the rest, since holding all dense waveforms would dominate memory.

In [ ]:
N_EVENTS = 5
INSPECT_IDX = 0  # event whose waveform we keep for plotting

wf_inspect = None
per_pmt_pe = []          # (n_events, n_channels)
per_event_total = []     # (n_events,)
n_vox_list = []

for ev in range(N_EVENTS):
    sw, n_raw, n_vox = simulate_event(ev)
    pe = sw.attrs.get('pe_counts')
    if pe is None:
        # Fallback: estimate from waveform energy / gain magnitude.
        wf = sw.deslice()
        pe = wf.adc.abs().sum(dim=1) / abs(GAIN)
    per_pmt_pe.append(pe.detach().cpu().numpy())
    per_event_total.append(float(pe.sum().item()))
    n_vox_list.append(n_vox)
    if ev == INSPECT_IDX:
        wf_inspect = sw.deslice()
    print(f'  event {ev:>2}: voxels={n_vox:>6,}  total PE={per_event_total[-1]:>12,.0f}')

per_pmt_pe = np.stack(per_pmt_pe)
print(f'\nstacked per-PMT PE shape: {per_pmt_pe.shape}')

## 5. Inspect the PE distribution

Left: per-PMT mean ± std PE across events (sanity-check that channels light up where you'd expect — half-detector symmetry visible at PMT 81). Right: per-event total PE.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

ax = axes[0]
mu = per_pmt_pe.mean(axis=0)
sd = per_pmt_pe.std(axis=0)
ch = np.arange(mu.size)
ax.bar(ch, mu, yerr=sd, color='tab:blue', alpha=0.85, ecolor='0.4', error_kw={'lw': 0.5})
ax.axvline(81, ls='--', lw=0.8, color='k', label='half-det boundary')
ax.set_xlabel('PMT id'); ax.set_ylabel('PE per event (mean ± std)')
ax.set_title(f'Per-PMT PE distribution across {N_EVENTS} events')
ax.set_yscale('symlog', linthresh=1.0)
ax.legend(fontsize=9)

ax = axes[1]
ax.bar(np.arange(N_EVENTS), per_event_total, color='tab:orange', alpha=0.85)
ax.set_xlabel('event index'); ax.set_ylabel('total PE')
ax.set_title('Per-event total PE yield')
for i, n in enumerate(n_vox_list):
    ax.text(i, per_event_total[i], f'N={n:,}', ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.show()

## 6. Inspect a single waveform

2-D `(PMT × time)` heatmap (cropped to the active window) plus the brightest PMT's 1-D trace.

In [ ]:
wf = wf_inspect
adc = wf.adc.cpu().numpy()
n_ch, n_bins = adc.shape

# crop time axis to a window around the global peak so the plot is readable
energy_per_bin = np.abs(adc).sum(0)
peak_bin = int(energy_per_bin.argmax())
PLOT_HALF = 1500
lo = max(0, peak_bin - PLOT_HALF); hi = min(n_bins, peak_bin + PLOT_HALF)
t_axis = wf.t0 + np.arange(lo, hi) * wf.tick_ns

ch_bright = int(np.abs(adc).sum(1).argmax())
vmax = float(np.abs(adc[:, lo:hi]).max())

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
ax = axes[0]
im = ax.imshow(
    adc[:, lo:hi], aspect='auto', cmap='RdBu_r', interpolation='none',
    norm=SymLogNorm(linthresh=1.0, linscale=1.0, vmin=-vmax, vmax=vmax),
    extent=[t_axis[0], t_axis[-1], 0, n_ch], origin='lower',
)
ax.axhline(81, ls='--', lw=0.6, color='k')
ax.set_xlabel('time (ns)'); ax.set_ylabel('PMT channel')
ax.set_title(f'event {INSPECT_IDX} waveform (cropped to {hi-lo} ticks of {n_bins})')
plt.colorbar(im, ax=ax, label='ADC')

ax = axes[1]
ax.plot(t_axis, adc[ch_bright, lo:hi], lw=1.0, color='k')
ax.set_xlabel('time (ns)'); ax.set_ylabel('ADC')
ax.set_title(f'PMT {ch_bright} (brightest in event {INSPECT_IDX})')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Benchmark — speed and peak GPU memory

Per-event measurement of `sim.simulate()` wall-clock (median of 3 runs after a warm-up) and `torch.cuda.max_memory_allocated()` peak. The voxelized inputs are pre-loaded so the timing isolates the goop pipeline (TOF → delays → histogram → FFT → digitize).

In [ ]:
BENCH_EVENTS = 5
REPEATS = 3

def time_and_peak(fn, n_warm=1):
    """Run fn() once to warm, then REPEATS times measuring time + peak mem."""
    for _ in range(n_warm):
        fn()
    torch.cuda.synchronize(); torch.cuda.empty_cache()
    times_ms, peaks_mb = [], []
    for _ in range(REPEATS):
        gc.collect(); torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = fn()
        torch.cuda.synchronize()
        times_ms.append((time.perf_counter() - t0) * 1000.0)
        peaks_mb.append(torch.cuda.max_memory_allocated() / (1024**2))
        del out
    return float(np.median(times_ms)), float(np.median(peaks_mb))

# Pre-load voxelized inputs so we time only the simulator step.
preloaded = []
for ev in range(BENCH_EVENTS):
    pos_t, nph_t, tns_t, n_raw, n_vox = load_voxelized(ev)
    preloaded.append((ev, n_raw, n_vox, pos_t, nph_t, tns_t))

print(f'{"event":>5} {"raw":>8} {"vox":>7} {"sim (ms)":>10} {"peak (MB)":>10}')
print('-' * 46)
rows = []
for ev, n_raw, n_vox, pos_t, nph_t, tns_t in preloaded:
    def _run(pos=pos_t, nph=nph_t, tns=tns_t):
        return sim.simulate(pos, nph, tns,
                            stitched=True, subtract_t0=True,
                            add_baseline_noise=False)
    t_ms, mb = time_and_peak(_run)
    rows.append((ev, n_raw, n_vox, t_ms, mb))
    print(f'{ev:>5} {n_raw:>8,} {n_vox:>7,} {t_ms:>9.1f} {mb:>10.1f}')

rows = np.array(rows, dtype=float)
print('-' * 46)
print(f'{"median":>5} {"":>8} {"":>7} {np.median(rows[:,3]):>9.1f} {np.median(rows[:,4]):>10.1f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(rows[:,2], rows[:,3], color='tab:blue'); axes[0].set_xlabel('N voxels'); axes[0].set_ylabel('sim wall-clock (ms)'); axes[0].set_title('Speed vs N')
axes[1].scatter(rows[:,2], rows[:,4], color='tab:red');  axes[1].set_xlabel('N voxels'); axes[1].set_ylabel('peak GPU memory (MB)'); axes[1].set_title('Memory vs N')
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()